# 04 — Risco de Violação de OLA

O objetivo aqui é: no instante em que um incidente entra na fila, estimar
a probabilidade de ele estourar o prazo — para que a operação consiga
intervir antes que aconteça, não depois.

Enfrentei dois desafios de verdade neste notebook: um **evento raro**
(apenas 0,95% da base viola OLA) e a **tentação constante de vazamento**,
já vista no notebook anterior.

In [ ]:
import sys, pathlib, warnings

# Descobre a pasta src/orion subindo a partir do diretório atual do kernel.
# Evita o erro "No module named 'orion'": o VS Code às vezes abre o notebook
# com o cwd na raiz do projeto, às vezes em notebooks/ — um caminho relativo
# fixo como '../src' só funciona no segundo caso.
_cwd = pathlib.Path.cwd()
for _base in [_cwd, *_cwd.parents]:
    _src = _base / 'src'
    if (_src / 'orion').is_dir():
        sys.path.insert(0, str(_src))
        break
else:
    raise FileNotFoundError(
        f"Não encontrei a pasta src/orion a partir de {_cwd}. "
        "Rode o notebook com o kernel na raiz do projeto (orion-aiops/) ou em notebooks/."
    )

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

In [ ]:
from orion.models.ola_model import (
    carregar, split_temporal, treinar, avaliar, curva_limiares, importancias, FEATURES,
)
from orion.config import PERCENTIL_FILA_RISCO

df = carregar()
print(f'{len(df):,} incidentes | {len(FEATURES)} features')
print(f"Positivos: {int(df['ola_violado'].sum())} ({df['ola_violado'].mean():.2%})")

## 1. Por que acurácia é a métrica errada aqui

Antes de escolher a métrica, fiz as contas: com 0,95% de eventos positivos,
um modelo bobo que responde "nunca viola" para tudo acerta 99,05% das vezes.
Um número ótimo — e um modelo completamente inútil, porque não previne
nenhuma violação. Isso me convenceu a nem cogitar acurácia como métrica
principal aqui.

In [ ]:
from sklearn.metrics import accuracy_score
y = df['ola_violado']
print(f'Acurácia do modelo "nunca viola": {accuracy_score(y, np.zeros(len(y))):.2%}')
print('Violações previstas por esse modelo: 0')
print('\nMétricas corretas para evento raro: PR-AUC, recall@k e lift sobre a prevalência.')

## 2. Validação temporal

Pela mesma razão do notebook de volume, um split aleatório aqui deixaria o
modelo aprender com o futuro. Separei os últimos 3 meses como teste, sem
misturar nenhuma data do teste no treino.

In [ ]:
treino, teste = split_temporal(df, meses_teste=3)
print(f'Treino: {len(treino):,} incidentes até {treino["dt_abertura"].max():%d/%m/%Y} '
      f'({int(treino["ola_violado"].sum())} positivos)')
print(f'Teste:  {len(teste):,} incidentes a partir de {teste["dt_abertura"].min():%d/%m/%Y} '
      f'({int(teste["ola_violado"].sum())} positivos)')

modelo = treinar(treino)
metricas, p = avaliar(modelo, teste)
pd.Series(metricas).to_frame('valor')

### Leitura das métricas

Aqui está como interpreto cada métrica que o modelo produziu:

* **ROC-AUC 0,84** — o modelo ordena bem: dado um par (violou, não violou), ele acerta
  qual é qual em 84% dos casos.
* **PR-AUC 0,19 contra prevalência de 0,96%** — dividindo um pelo outro dá um **lift**
  de ~19x. Lift é só "quantas vezes melhor que o acaso": se eu escolhesse incidentes
  aleatoriamente, acertaria 0,96% das vezes; com o modelo, a taxa de acerto sobe pra
  quase 19x mais que isso. Esse é o número que realmente importa quando o evento é
  raro, mais do que o ROC-AUC.
* **Brier 0,009** — as probabilidades estão bem calibradas, ou seja, quando o modelo
  diz "30% de chance", isso de fato se aproxima da frequência real.

Guardei uma regra do notebook anterior: um AUC de 0,97 aqui seria sinal de
vazamento, não de excelência. Como fiquei em 0,84, isso me dá mais confiança
de que o resultado é real.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc

yte = teste['ola_violado'].values
fpr, tpr, _ = roc_curve(yte, p)
prec, rec, _ = precision_recall_curve(yte, p)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(fpr, tpr, color='#38BDF8', lw=2, label=f'ORION (AUC={auc(fpr,tpr):.3f})')
axes[0].plot([0, 1], [0, 1], ls='--', color='gray', label='Acaso')
axes[0].set_xlabel('Falso positivo'); axes[0].set_ylabel('Verdadeiro positivo')
axes[0].set_title('Curva ROC'); axes[0].legend()

axes[1].plot(rec, prec, color='#38BDF8', lw=2)
axes[1].axhline(yte.mean(), ls='--', color='#EF4444',
                label=f'Prevalência ({yte.mean():.2%})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precisão')
axes[1].set_title('Curva Precisão-Recall'); axes[1].legend()
plt.tight_layout(); plt.show()

## 3. O ponto de operação — a decisão de negócio

Percebi rápido que um limiar de 0,5 não alertaria nada, dado o
desbalanceamento. Em vez de escolher um número arbitrário, decidi definir a
regra em termos que o gestor da operação realmente negocia: **"quantos
chamados minha equipe consegue revisar por dia?"**

In [ ]:
dias = teste['data'].nunique()
linhas = []
for pct in [1, 3, 5, 10, 15, 20, 30]:
    limiar = np.quantile(p, 1 - pct/100)
    flag = p >= limiar
    linhas.append({
        'top %': f'{pct}%',
        'alertas/dia': round(flag.sum() / dias, 1),
        'recall': f'{yte[flag].sum() / yte.sum():.0%}',
        'precisão': f'{yte[flag].mean():.1%}',
        'lift': round(yte[flag].mean() / yte.mean(), 1),
    })
pd.DataFrame(linhas)

In [ ]:
pcts = np.arange(1, 41)
recalls, cargas = [], []
for pct in pcts:
    limiar = np.quantile(p, 1 - pct/100)
    flag = p >= limiar
    recalls.append(yte[flag].sum() / yte.sum() * 100)
    cargas.append(flag.sum() / dias)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(cargas, recalls, color='#38BDF8', lw=2.5)
idx = list(pcts).index(10)
ax.scatter([cargas[idx]], [recalls[idx]], s=160, color='#EF4444', zorder=5)
ax.annotate(f'Recomendado: top 10%\n{cargas[idx]:.1f} revisões/dia\ncaptura {recalls[idx]:.0f}% das violações',
            xy=(cargas[idx], recalls[idx]), xytext=(cargas[idx]+3, recalls[idx]-18),
            arrowprops=dict(arrowstyle='->', color='#EF4444'), color='#EF4444', fontweight='bold')
ax.set_xlabel('revisões manuais por dia'); ax.set_ylabel('% das violações capturadas')
ax.set_title('Curva de custo-benefício operacional')
plt.tight_layout(); plt.show()

**Ponto de operação que escolhi: top 10% da fila diária.**

Com ~6 revisões por dia, capturo 52% das violações de OLA. Traduzindo para
o contrato: em 2025, isso significaria a chance de interceptar ~22 das 42
violações de P2 — mais do que suficiente para devolver o indicador da
faixa de 75% para a de 100%, exatamente o achado que motivou este modelo
lá no notebook 01.

## 4. Explicabilidade

Com o ponto de operação definido, quero entender o que o modelo está
enxergando — não basta confiar no número, preciso conseguir explicar a
decisão para quem vai usar o sistema no dia a dia.

In [ ]:
imp = importancias(modelo)
fig, ax = plt.subplots(figsize=(10, 7))
imp.head(15).sort_values('importancia_pct').plot(
    x='feature', y='importancia_pct', kind='barh', ax=ax, color='#38BDF8', legend=False)
ax.set_title('Importância das features — risco de OLA')
ax.set_xlabel('importância relativa'); ax.set_ylabel('')
plt.tight_layout(); plt.show()
imp.head(12)

Vi que as features de **histórico de violação** (grupo, IC, categoria, produto)
e de **carga operacional no instante** dominam a importância. Isso faz sentido
do ponto de vista operacional: o que melhor prevê um atraso é quem está
atendendo, em qual ativo, e sob qual pressão — não um atributo isolado do
chamado.

In [ ]:
# Análise por segmento — onde o modelo acerta mais
teste_ = teste.copy()
teste_['score'] = p

# Corte de top 10% calculado DENTRO de cada prioridade, não na fila inteira.
# Com um corte único, P3 (que tem 3x mais chamados no teste) dominava o
# ranking e quase nenhum P2 entrava no top 10% global — mesmo P2 sendo a
# prioridade que este projeto existe para proteger. Calculando o corte por
# grupo, cada prioridade disputa a vaga só com ela mesma.
teste_['no_top10'] = teste_.groupby('prioridade')['score'].transform(
    lambda s: s >= s.quantile(0.9)
)

por_prio = teste_.groupby('prioridade').agg(
    incidentes=('score', 'size'),
    violacoes=('ola_violado', 'sum'),
    capturadas=('ola_violado', lambda s: int((s & teste_.loc[s.index, 'no_top10']).sum())),
)
por_prio['recall'] = (por_prio['capturadas'] / por_prio['violacoes']).map('{:.0%}'.format)
por_prio

In [ ]:
from orion.models.ola_model import executar
modelo_final, metricas_finais = executar()
print('Artefatos gravados: risco_ola_scores, curva_limiar_ola, importancia_ola')
print(f"ROC-AUC {metricas_finais['roc_auc']:.3f} | recall@top10% {metricas_finais['recall_top_10pct']:.0%}")

Com os dois modelos prontos, sigo para a etapa que transforma número em
ação: `05_agentes_orquestracao.ipynb`.